# Garuda multi-query search → RIS

Runs several searches on [Garuda](https://garuda.kemdiktisaintek.go.id) (Garba Rujukan Digital), merges the results and saves:

| File (in `garuda_output/`) | What it is |
| --- | --- |
| `garuda_deduplicated.ris` | All records, duplicates merged. Import this into Zotero / Mendeley / Rayyan. |
| `garuda_all_records.ris` | All records before duplicates are merged (one per Garuda ID). |
| `ris_per_query/A1.ris`, `A2.ris`, … | The records each query found. |
| `garuda_screening.xlsx` / `.csv` | One row per record, with a `found_by` column (e.g. `A1:author; A2:author`). |
| `search_log.csv` | Per search: text sent, field, Garuda's reported total, records retrieved, status, time. For a PRISMA search log. |

The example collects the publications of two authors. Replace `QUERIES` and `FIELDS` in section 2 with your own searches, for example topic keywords searched in `title` and `abstract` for a literature review.

**To run:** install `garuda2ris` (see the README), then *Run All*. Finished searches are cached, so running again only repeats what failed.

## 1. Setup

In [ ]:
import sys, subprocess, importlib
from importlib import metadata
from pathlib import Path

REQUIRED = (0, 2, 0)

def _installed():
    try:
        return tuple(int(x) for x in metadata.version("garuda2ris").split(".")[:3])
    except metadata.PackageNotFoundError:
        return None

if (_installed() or (0, 0, 0)) < REQUIRED:
    places = [Path.cwd(), Path.home() / "Downloads", Path.home() / "Desktop"]
    found = [z for place in places for z in sorted(place.glob("garuda2ris-0.2*.zip"))]
    if not found:
        raise FileNotFoundError(
            "garuda2ris 0.2.0 or newer is not installed. Install it as the README describes "
            "(pip install ...), or put garuda2ris-0.2.0.zip next to this notebook, then run this cell again."
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", str(found[0])])
    importlib.invalidate_caches()
    for name in [m for m in sys.modules if m.split(".")[0] == "garuda2ris"]:
        del sys.modules[name]

try:
    import pandas as pd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "pandas", "openpyxl"])
    import pandas as pd

import json, logging, re
from collections import defaultdict
from datetime import datetime

import garuda2ris
from garuda2ris import (
    Article, GarudaClient, GarudaError,
    dedupe_groups, extract_keywords, normalize_query, write_ris,
)

logging.basicConfig(level=logging.INFO, format="   %(message)s", stream=sys.stdout, force=True)
pd.set_option("display.max_colwidth", 90)
print("garuda2ris", garuda2ris.__version__)

## 2. Searches and settings

In [ ]:
QUERIES = {
    "A1": "Cendra Devayana Putra",
    "A2": "Bartolomeus Priya",
}

FIELDS = ["author"]                # Garuda searches one field at a time: "title", "abstract", "author", "doi"
NORMALIZE = True                   # rewrite hyphens and quotes before sending (see the note below)

YEAR_FROM, YEAR_TO = None, None    # e.g. 2015, 2025. Records with no year shown are kept.
REMOVE_DUPLICATES = True           # merge records that share a DOI or a title
EXTRACT_KEYWORDS = True            # copy "Keywords: ..." at the end of abstracts into RIS KW fields

OUT_DIR = Path("garuda_output")
DELAY = 1.5                        # seconds between requests; please keep it at 1 or more
MAX_RECORDS_PER_SEARCH = None      # safety cap per search, e.g. 500
USE_CACHE = True                   # reuse searches that already finished

### How Garuda reads a query

Garuda's search box does not work like Scopus (checked on the live site, October 2026):

- **All words must be present, in any order.** There is no phrase search, and quotes change nothing.
- **A hyphen inside a word excludes what follows it.** `smoke-free` returns records that contain *smoke* and do **not** contain *free*.
- **Very short words seem to be ignored.**

With `NORMALIZE = True` each query is therefore sent without quotes and with hyphens turned into spaces. The original wording is kept in the search log next to the text actually sent.

To search a topic in both title and abstract, set `FIELDS = ["title", "abstract"]`: each query is then run once per field and the results are merged.

In [ ]:
preview = pd.DataFrame(
    [{"id": qid, "as written": q, "sent to Garuda": normalize_query(q) if NORMALIZE else q}
     for qid, q in QUERIES.items()]
)
display(preview)

## 3. Crawl

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUT_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)

client = GarudaClient(delay=DELAY)
hits = {}        # (query id, field) -> list of Article
log_rows = []

for qid, query in QUERIES.items():
    sent = normalize_query(query) if NORMALIZE else query
    for field in FIELDS:
        cache_file = CACHE_DIR / f"{qid}_{field}.json"
        row = {"id": qid, "query": query, "sent_to_garuda": sent, "field": field}

        saved = None
        if USE_CACHE and cache_file.exists():
            saved = json.loads(cache_file.read_text(encoding="utf-8"))
            if saved.get("sent_to_garuda") != sent:
                saved = None

        if saved:
            articles = [Article(**d) for d in saved["records"]]
            row.update(garuda_reports=saved["garuda_reports"], retrieved=len(articles),
                       status="complete (cached)", searched_at=saved["searched_at"])
            print(f"{qid} [{field}]  {sent}  ->  {len(articles)} records (cached)")
        else:
            print(f"{qid} [{field}]  {sent}")
            stamp = datetime.now().isoformat(timespec="seconds")
            reported = None
            try:
                articles = list(client.search(sent, field=field, normalize=False,
                                              max_records=MAX_RECORDS_PER_SEARCH))
                reported = client.total_records
                if reported is not None and len(articles) == reported:
                    status = "complete"
                elif MAX_RECORDS_PER_SEARCH and len(articles) >= MAX_RECORDS_PER_SEARCH:
                    status = f"capped at {MAX_RECORDS_PER_SEARCH}"
                else:
                    status = "INCOMPLETE"
            except GarudaError as exc:
                articles, status = [], f"FAILED: {exc}"
            if status == "complete":
                cache_file.write_text(json.dumps(
                    {"sent_to_garuda": sent, "field": field, "garuda_reports": reported,
                     "searched_at": stamp, "records": [a.to_dict() for a in articles]},
                    ensure_ascii=False), encoding="utf-8")
            row.update(garuda_reports=reported, retrieved=len(articles), status=status, searched_at=stamp)
            print(f"   -> {len(articles)} of {reported if reported is not None else '?'} records, {status}")

        hits[(qid, field)] = articles
        log_rows.append(row)

## 4. Search log

`garuda_reports` is the total Garuda shows for the search; `retrieved` is what was downloaded. Anything not marked *complete* should be run again (just re-run section 3: finished searches come from the cache).

In [ ]:
search_log = pd.DataFrame(log_rows)
search_log.to_csv(OUT_DIR / "search_log.csv", index=False, encoding="utf-8-sig")
display(search_log.drop(columns=["query"]))

not_ok = search_log[~search_log["status"].str.startswith("complete")]
if len(not_ok):
    print(f"\n{len(not_ok)} of {len(search_log)} searches did not finish cleanly. Re-run section 3.")
    print("If it keeps failing, please contact Cendra Devayana Putra: "
          "https://github.com/Cendra123/garuda2ris/issues")
else:
    print(f"\nAll {len(search_log)} searches complete.")

## 5. Combine, filter and remove duplicates

In [ ]:
# one record per Garuda ID, remembering which searches found it
by_id, found_by = {}, defaultdict(list)
for (qid, field), articles in hits.items():
    for a in articles:
        by_id.setdefault(a.garuda_id, a)
        found_by[a.garuda_id].append(f"{qid}:{field}")

records = list(by_id.values())

if EXTRACT_KEYWORDS:
    for a in records:
        if not a.keywords:
            a.keywords = extract_keywords(a.abstract)

def in_year_range(a):
    if a.year is None:
        return True
    return (YEAR_FROM is None or a.year >= YEAR_FROM) and (YEAR_TO is None or a.year <= YEAR_TO)

records = [a for a in records if in_year_range(a)]

# the same paper is often listed under two Garuda IDs: merge those
groups = dedupe_groups(records) if REMOVE_DUPLICATES else [[a] for a in records]
order = {qid: i for i, qid in enumerate(QUERIES)}
unique, sources, merged_ids = [], {}, {}
for group in groups:
    first = group[0]
    for other in group[1:]:
        for name, value in other.to_dict().items():
            if value and not getattr(first, name):
                setattr(first, name, value)
    unique.append(first)
    tags = {tag for a in group for tag in found_by[a.garuda_id]}
    sources[first.garuda_id] = sorted(
        tags, key=lambda t: (order[t.split(":")[0]], FIELDS.index(t.split(":")[1])))
    merged_ids[first.garuda_id] = [a.garuda_id for a in group[1:]]

total_hits = sum(len(v) for v in hits.values())
print(f"Records retrieved over all {len(hits)} searches : {total_hits}")
print(f"Unique Garuda records                  : {len(by_id)}")
if YEAR_FROM is not None or YEAR_TO is not None:
    print(f"After year filter {YEAR_FROM or ''}-{YEAR_TO or ''}            : {len(records)}")
print(f"After merging duplicate records        : {len(unique)}")

kept_ids = {a.garuda_id for a in records}
rows = []
for qid in QUERIES:
    mine = {a.garuda_id for f in FIELDS for a in hits.get((qid, f), [])} & kept_ids
    others = {a.garuda_id for (q, f), arts in hits.items() if q != qid for a in arts}
    rows.append({"id": qid, "sent to Garuda": normalize_query(QUERIES[qid]) if NORMALIZE else QUERIES[qid],
                 **{f: len(hits.get((qid, f), [])) for f in FIELDS},
                 "unique records": len(mine), "found by this query only": len(mine - others)})
per_query = pd.DataFrame(rows)
display(per_query)

## 6. Export

In [ ]:
RIS_DIR = OUT_DIR / "ris_per_query"
RIS_DIR.mkdir(exist_ok=True)

for qid in QUERIES:
    seen, subset = set(), []
    for f in FIELDS:
        for a in hits.get((qid, f), []):
            if a.garuda_id in kept_ids and a.garuda_id not in seen:
                seen.add(a.garuda_id)
                subset.append(by_id[a.garuda_id])
    write_ris(subset, RIS_DIR / f"{qid}.ris")

write_ris(records, OUT_DIR / "garuda_all_records.ris")
write_ris(unique, OUT_DIR / "garuda_deduplicated.ris")

def _clean(value):
    return re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", " ", value) if isinstance(value, str) else value

COLUMNS = ["garuda_id", "found_by", "title", "authors", "year", "journal", "volume", "issue",
           "publisher", "doi", "url", "garuda_url", "abstract", "keywords", "duplicate_garuda_ids"]
screening = pd.DataFrame([{
    "garuda_id": a.garuda_id,
    "found_by": "; ".join(sources[a.garuda_id]),
    "title": a.title,
    "authors": "; ".join(a.authors),
    "year": a.year,
    "journal": a.journal,
    "volume": a.volume,
    "issue": a.issue,
    "publisher": a.publisher,
    "doi": a.doi,
    "url": a.url,
    "garuda_url": a.garuda_url,
    "abstract": a.abstract,
    "keywords": "; ".join(a.keywords),
    "duplicate_garuda_ids": "; ".join(merged_ids[a.garuda_id]),
} for a in unique], columns=COLUMNS).apply(lambda col: col.map(_clean))
screening["year"] = screening["year"].astype("Int64")

screening.to_csv(OUT_DIR / "garuda_screening.csv", index=False, encoding="utf-8-sig")
try:
    screening.to_excel(OUT_DIR / "garuda_screening.xlsx", index=False)
except ImportError:
    print("openpyxl is not installed, so no .xlsx was written (the .csv has the same content).")
per_query.to_csv(OUT_DIR / "records_per_query.csv", index=False, encoding="utf-8-sig")

print("Saved in", OUT_DIR.resolve())
for path in sorted(p for p in OUT_DIR.rglob("*") if p.is_file() and "cache" not in p.parts):
    print(f"  {str(path.relative_to(OUT_DIR)):38} {path.stat().st_size / 1024:8.1f} KB")

## 7. Quick look

In [ ]:
display(screening[["found_by", "year", "title", "journal"]].head(15))

by_year = screening["year"].dropna().value_counts().sort_index()
display(by_year.rename("records").to_frame().T)
print("Records with no year shown by Garuda:", int(screening["year"].isna().sum()))